# 08 — Epsilon Sensitivity: Country-Level Continuous Dominance

This notebook comes after:

`07_Profile_POSet_Main.ipynb`

Purpose:

Test how country-level dominance relations change when an epsilon tolerance is introduced.

This notebook does:

- read the direction-aligned Pre-POSet snapshot;
- use continuous scaled variables, not ordinal profile codes;
- run country-level dominance checks by shock and variable set;
- test an epsilon grid;
- detect when epsilon creates cycles or reciprocal dominance;
- identify Pareto/frontier countries across epsilon values;
- export dominance relations, Pareto countries, comparability metrics, cycle diagnostics, and frontier stability summaries.

This notebook does **not**:

- replace the main profile-level POSet;
- create the final Hasse diagram for the report;
- use recovery as an ordering variable;
- decide the final epsilon value;
- make a final scalar resilience ranking.

Important:

The epsilon rule in this notebook is a **diagnostic tolerance rule**:

```text
A epsilon-dominates B if:
A + epsilon >= B in every dimension
and A > B + epsilon in at least one dimension.
```

This rule can create cycles, so every epsilon run is checked for partial-order validity.

The safer epsilon-margin robustness method is kept for:

`09_Epsilon_Margin_POSet_Robustness.ipynb`

In [1]:
# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import re
import shutil
import warnings
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 350)
pd.set_option("display.max_rows", 250)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

PRE_POSET_DIR = PROJECT_ROOT / "Data" / "Processed" / "Pre_POSet_EDA"

INPUT_SNAPSHOT_FILE = PRE_POSET_DIR / "pre_poset_analysis_ready_snapshot_full.csv"
CANDIDATE_VARIABLE_SETS_FILE = PRE_POSET_DIR / "candidate_variable_sets.csv"
BASELINE_COMPLETE_CASE_FILE = PRE_POSET_DIR / "baseline_complete_case_sample_by_shock.csv"

PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed" / "Epsilon_Sensitivity_Country_Level"
DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "08_Epsilon_Sensitivity_Country_Level"
FIGURES_DIR = PROJECT_ROOT / "Outputs" / "Figures" / "Epsilon_Sensitivity_Country_Level"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

CLEAR_PREVIOUS_OUTPUTS = True

if CLEAR_PREVIOUS_OUTPUTS:
    for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, FIGURES_DIR]:
        if folder.exists():
            shutil.rmtree(folder)

for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, FIGURES_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Run ID:", RUN_ID)
print("Input snapshot file:", INPUT_SNAPSHOT_FILE.resolve())
print("Candidate variable sets file:", CANDIDATE_VARIABLE_SETS_FILE.resolve())
print("Processed folder:", PROCESSED_DIR.resolve())
print("Diagnostics folder:", DIAGNOSTICS_DIR.resolve())
print("Figures folder:", FIGURES_DIR.resolve())

Run ID: 20260622_080229
Input snapshot file: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Pre_POSet_EDA\pre_poset_analysis_ready_snapshot_full.csv
Candidate variable sets file: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Pre_POSet_EDA\candidate_variable_sets.csv
Processed folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Epsilon_Sensitivity_Country_Level
Diagnostics folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Diagnostics\08_Epsilon_Sensitivity_Country_Level
Figures folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Outputs\Figures\Epsilon_Sensitivity_Country_Level


In [2]:
# ------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------

SHOCK_IDS = ["2007", "2019"]

MAIN_SET_NAME = "baseline_6_variables"

# Epsilon values are applied after variables are scaled to [0, 1] within each run.
# Example: epsilon = 0.05 means 5 percentage points on the run-specific 0-1 scale.
EPSILON_GRID = [0.00, 0.02, 0.05, 0.10, 0.15, 0.20]

# Run all candidate sets from notebook 06, but keep the working-main summaries separate.
RUN_ALL_CANDIDATE_SETS = True

# Complete-case by selected ordering variables.
SAMPLE_STRATEGY = "complete_case"

# Keep Recovery available only for later validation interpretation, never as an ordering variable.
REQUIRE_RECOVERY_AVAILABLE = True

# If a candidate set has more variables than this, it is still run,
# but later reports should be careful about dimensionality.
MAX_RECOMMENDED_VARIABLE_COUNT = 6

FALLBACK_BASELINE_VARIABLES = [
    "debt_capacity_score_0_100",
    "employment_strength_score_0_100",
    "rd_intensity_score_0_100",
    "human_capital_tertiary_score_0_100",
    "energy_security_score_0_100",
    "governance_capacity_score_0_100",
]

print("Epsilon grid:", EPSILON_GRID)
print("Main set:", MAIN_SET_NAME)
print("Sample strategy:", SAMPLE_STRATEGY)
print("Require recovery available:", REQUIRE_RECOVERY_AVAILABLE)

Epsilon grid: [0.0, 0.02, 0.05, 0.1, 0.15, 0.2]
Main set: baseline_6_variables
Sample strategy: complete_case
Require recovery available: True


In [3]:
# ------------------------------------------------------
# HELPER FUNCTIONS
# ------------------------------------------------------

def require_file(path, label):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")
    return path


def clean_country_shock(df):
    out = df.copy()

    if "Country" in out.columns:
        out["Country"] = out["Country"].astype(str).str.strip().str.upper()

    if "shock_id" in out.columns:
        out["shock_id"] = out["shock_id"].astype(str).str.strip()

    return out


def parse_variable_list(value):
    if pd.isna(value):
        return []

    return [
        item.strip()
        for item in str(value).split(",")
        if item.strip()
    ]


def variable_applicable_to_shock(variable, shock_id):
    variable = str(variable)
    shock_id = str(shock_id)

    match = re.search(r"_pre_(\d{4})", variable)

    if match:
        return match.group(1) == shock_id

    return True


def min_max_scale_to_unit_interval(series):
    values = pd.to_numeric(series, errors="coerce")
    min_value = values.min(skipna=True)
    max_value = values.max(skipna=True)

    if pd.isna(min_value) or pd.isna(max_value):
        return pd.Series(np.nan, index=series.index)

    if np.isclose(min_value, max_value):
        return pd.Series(0.5, index=series.index)

    return (values - min_value) / (max_value - min_value)


def build_epsilon_tolerance_dominance_matrix(values, epsilon):
    # values are assumed direction-aligned and scaled to [0, 1].
    matrix = np.asarray(values, dtype=float)
    n = matrix.shape[0]
    dominance = np.zeros((n, n), dtype=bool)

    for i in range(n):
        for j in range(n):
            if i == j:
                continue

            # Diagnostic tolerance rule:
            # A may be up to epsilon worse in any dimension,
            # but must be more than epsilon better in at least one dimension.
            all_not_much_worse = np.all(matrix[i, :] + epsilon >= matrix[j, :])
            meaningful_advantage = np.any(matrix[i, :] > matrix[j, :] + epsilon)

            if all_not_much_worse and meaningful_advantage:
                dominance[i, j] = True

    return dominance


def build_strict_dominance_matrix(values):
    matrix = np.asarray(values, dtype=float)
    n = matrix.shape[0]
    dominance = np.zeros((n, n), dtype=bool)

    for i in range(n):
        for j in range(n):
            if i == j:
                continue

            ge_all = np.all(matrix[i, :] >= matrix[j, :])
            gt_any = np.any(matrix[i, :] > matrix[j, :])

            if ge_all and gt_any:
                dominance[i, j] = True

    return dominance


def detect_reciprocal_pairs(dominance):
    reciprocal_pairs = []

    n = dominance.shape[0]

    for i in range(n):
        for j in range(i + 1, n):
            if dominance[i, j] and dominance[j, i]:
                reciprocal_pairs.append((i, j))

    return reciprocal_pairs


def is_dag_kahn(adjacency):
    n = adjacency.shape[0]
    indegree = adjacency.sum(axis=0).astype(int)
    queue = [i for i in range(n) if indegree[i] == 0]
    visited = 0

    while queue:
        node = queue.pop(0)
        visited += 1

        for target in np.where(adjacency[node, :])[0]:
            indegree[target] -= 1

            if indegree[target] == 0:
                queue.append(int(target))

    return visited == n


def assign_layers_if_dag(dominance):
    if not is_dag_kahn(dominance):
        return None

    n = dominance.shape[0]
    remaining = set(range(n))
    layers = {i: np.nan for i in range(n)}
    layer = 1

    while remaining:
        nondominated = []

        for i in remaining:
            dominated_by_remaining = any(
                dominance[j, i]
                for j in remaining
                if j != i
            )

            if not dominated_by_remaining:
                nondominated.append(i)

        if not nondominated:
            return None

        for i in nondominated:
            layers[i] = layer

        remaining -= set(nondominated)
        layer += 1

    return layers


def transitive_reduction_dag(adjacency):
    # Works for DAGs. Removes edge i->j if there is an alternative path i -> ... -> j.
    n = adjacency.shape[0]
    reach = adjacency.copy().astype(bool)

    # Warshall transitive closure.
    for k in range(n):
        reach = reach | (reach[:, [k]] & reach[[k], :])

    reduced_edges = []

    for i in range(n):
        for j in range(n):
            if not adjacency[i, j]:
                continue

            has_alternative_path = False

            for k in range(n):
                if k == i or k == j:
                    continue

                if adjacency[i, k] and reach[k, j]:
                    has_alternative_path = True
                    break

            if not has_alternative_path:
                reduced_edges.append((i, j))

    return reduced_edges


def comparability_metrics(dominance):
    n = dominance.shape[0]
    total_pairs = n * (n - 1) / 2

    if total_pairs == 0:
        return {
            "country_count": n,
            "comparable_pairs": 0,
            "incomparable_pairs": 0,
            "comparability_ratio": np.nan,
            "incomparability_ratio": np.nan,
        }

    comparable_pairs = 0

    for i in range(n):
        for j in range(i + 1, n):
            if dominance[i, j] or dominance[j, i]:
                comparable_pairs += 1

    incomparable_pairs = int(total_pairs - comparable_pairs)

    return {
        "country_count": n,
        "comparable_pairs": comparable_pairs,
        "incomparable_pairs": incomparable_pairs,
        "comparability_ratio": comparable_pairs / total_pairs,
        "incomparability_ratio": incomparable_pairs / total_pairs,
    }


def jaccard_similarity(set_a, set_b):
    set_a = set(set_a)
    set_b = set(set_b)

    if not set_a and not set_b:
        return np.nan

    return len(set_a & set_b) / len(set_a | set_b)


def safe_sheet_name(name, used):
    cleaned = re.sub(r"[/\\*\[\]:?]", "_", str(name))[:31]
    base = cleaned
    i = 1

    while cleaned in used:
        suffix = f"_{i}"
        cleaned = base[:31 - len(suffix)] + suffix
        i += 1

    used.add(cleaned)
    return cleaned


def estimate_width(series, col_name, min_width=10, max_width=60):
    values = [str(col_name)] + series.dropna().astype(str).head(200).tolist()

    if not values:
        return min_width

    return max(min_width, min(max(len(v) for v in values) + 2, max_width))

In [4]:
# ------------------------------------------------------
# LOAD INPUTS
# ------------------------------------------------------

require_file(INPUT_SNAPSHOT_FILE, "Pre-POSet analysis-ready snapshot")

snapshot = pd.read_csv(INPUT_SNAPSHOT_FILE)
snapshot = clean_country_shock(snapshot)

if "country_name" not in snapshot.columns:
    snapshot["country_name"] = snapshot["Country"]

required_cols = {"Country", "shock_id", "Recovery"}
missing_required = required_cols - set(snapshot.columns)

if missing_required:
    raise ValueError(f"Input snapshot missing required columns: {missing_required}")

if CANDIDATE_VARIABLE_SETS_FILE.exists():
    candidate_variable_sets = pd.read_csv(CANDIDATE_VARIABLE_SETS_FILE)
else:
    candidate_variable_sets = pd.DataFrame([{
        "set_name": MAIN_SET_NAME,
        "intended_use": "main_candidate",
        "shock_specific": False,
        "variables": ",".join(FALLBACK_BASELINE_VARIABLES),
        "variable_count": len(FALLBACK_BASELINE_VARIABLES),
        "note": "Fallback baseline set because candidate_variable_sets.csv was not found.",
    }])

if not RUN_ALL_CANDIDATE_SETS:
    candidate_variable_sets = candidate_variable_sets[
        candidate_variable_sets["set_name"] == MAIN_SET_NAME
    ].copy()

if BASELINE_COMPLETE_CASE_FILE.exists():
    baseline_complete_case_sample = pd.read_csv(BASELINE_COMPLETE_CASE_FILE)
    baseline_complete_case_sample = clean_country_shock(baseline_complete_case_sample)
else:
    baseline_complete_case_sample = pd.DataFrame()

input_summary = pd.DataFrame([{
    "rows": len(snapshot),
    "countries": snapshot["Country"].nunique(),
    "shock_ids": ",".join(sorted(snapshot["shock_id"].unique())),
    "candidate_variable_sets": len(candidate_variable_sets),
    "baseline_complete_case_rows": len(baseline_complete_case_sample),
    "run_timestamp": RUN_TIMESTAMP,
}])

input_summary.to_csv(
    DIAGNOSTICS_DIR / "epsilon_input_summary.csv",
    index=False,
)

print("Snapshot rows:", len(snapshot))
print("Countries:", snapshot["Country"].nunique())
print("Shock IDs:", sorted(snapshot["shock_id"].unique()))
print("Candidate variable sets:")
display(candidate_variable_sets)

Snapshot rows: 300
Countries: 162
Shock IDs: ['2007', '2019']
Candidate variable sets:


,set_name,intended_use,shock_specific,variables,variable_count,note
0,baseline_6_variables,main_candidate,False,"debt_capacity_score_0_100,employment_strength_...",6,"Balanced structural set across fiscal, labour,..."
1,baseline_plus_gdp_stability_2007,shock_2007_sensitivity,True,"debt_capacity_score_0_100,employment_strength_...",7,Adds pre-2007 GDP growth stability. Use only f...
2,baseline_plus_gdp_stability_2019,shock_2019_sensitivity,True,"debt_capacity_score_0_100,employment_strength_...",7,Adds pre-2019 GDP growth stability. Use only f...
3,sensitivity_remove_governance,sensitivity,False,"debt_capacity_score_0_100,employment_strength_...",5,Tests dependence on derived governance composite.
4,sensitivity_replace_debt_with_productivity,sensitivity,False,"employment_strength_score_0_100,rd_intensity_s...",6,Tests whether productive capacity changes orde...


In [5]:
# ------------------------------------------------------
# BUILD EPSILON RUN PLAN
# ------------------------------------------------------

run_plan_rows = []

for _, row in candidate_variable_sets.iterrows():
    set_name = str(row["set_name"])
    variables = parse_variable_list(row["variables"])
    intended_use = row.get("intended_use", "")
    shock_specific = bool(row.get("shock_specific", False))
    note = row.get("note", "")

    if not variables:
        continue

    if set_name.endswith("_2007"):
        shocks_for_set = ["2007"]
    elif set_name.endswith("_2019"):
        shocks_for_set = ["2019"]
    else:
        shocks_for_set = SHOCK_IDS

    for shock_id in shocks_for_set:
        applicable_variables = [
            var for var in variables
            if variable_applicable_to_shock(var, shock_id)
        ]

        missing_variables = [
            var for var in applicable_variables
            if var not in snapshot.columns
        ]

        for epsilon in EPSILON_GRID:
            run_plan_rows.append({
                "run_key": f"{set_name}__shock_{shock_id}__eps_{epsilon:.2f}",
                "set_name": set_name,
                "intended_use": intended_use,
                "shock_id": shock_id,
                "epsilon": float(epsilon),
                "variables": ",".join(applicable_variables),
                "variable_count": len(applicable_variables),
                "missing_variables": ",".join(missing_variables),
                "missing_variable_count": len(missing_variables),
                "sample_strategy": SAMPLE_STRATEGY,
                "require_recovery_available": REQUIRE_RECOVERY_AVAILABLE,
                "is_working_main_set": set_name == MAIN_SET_NAME,
                "note": note,
            })

epsilon_run_plan = pd.DataFrame(run_plan_rows)

epsilon_run_plan.to_csv(
    PROCESSED_DIR / "epsilon_run_plan.csv",
    index=False,
)

epsilon_run_plan.to_csv(
    DIAGNOSTICS_DIR / "epsilon_run_plan.csv",
    index=False,
)

display(epsilon_run_plan)

,run_key,set_name,intended_use,shock_id,epsilon,variables,variable_count,missing_variables,missing_variable_count,sample_strategy,require_recovery_available,is_working_main_set,note
0,baseline_6_variables__shock_2007__eps_0.00,baseline_6_variables,main_candidate,2007,0.0000,"debt_capacity_score_0_100,employment_strength_...",6,,0,complete_case,True,True,"Balanced structural set across fiscal, labour,..."
1,baseline_6_variables__shock_2007__eps_0.02,baseline_6_variables,main_candidate,2007,0.0200,"debt_capacity_score_0_100,employment_strength_...",6,,0,complete_case,True,True,"Balanced structural set across fiscal, labour,..."
2,baseline_6_variables__shock_2007__eps_0.05,baseline_6_variables,main_candidate,2007,0.0500,"debt_capacity_score_0_100,employment_strength_...",6,,0,complete_case,True,True,"Balanced structural set across fiscal, labour,..."
3,baseline_6_variables__shock_2007__eps_0.10,baseline_6_variables,main_candidate,2007,0.1000,"debt_capacity_score_0_100,employment_strength_...",6,,0,complete_case,True,True,"Balanced structural set across fiscal, labour,..."
4,baseline_6_variables__shock_2007__eps_0.15,baseline_6_variables,main_candidate,2007,0.1500,"debt_capacity_score_0_100,employment_strength_...",6,,0,complete_case,True,True,"Balanced structural set across fiscal, labour,..."
5,baseline_6_variables__shock_2007__eps_0.20,baseline_6_variables,main_candidate,2007,0.2000,"debt_capacity_score_0_100,employment_strength_...",6,,0,complete_case,True,True,"Balanced structural set across fiscal, labour,..."
6,baseline_6_variables__shock_2019__eps_0.00,baseline_6_variables,main_candidate,2019,0.0000,"debt_capacity_score_0_100,employment_strength_...",6,,0,complete_case,True,True,"Balanced structural set across fiscal, labour,..."
7,baseline_6_variables__shock_2019__eps_0.02,baseline_6_variables,main_candidate,2019,0.0200,"debt_capacity_score_0_100,employment_strength_...",6,,0,complete_case,True,True,"Balanced structural set across fiscal, labour,..."
8,baseline_6_variables__shock_2019__eps_0.05,baseline_6_variables,main_candidate,2019,0.0500,"debt_capacity_score_0_100,employment_strength_...",6,,0,complete_case,True,True,"Balanced structural set across fiscal, labour,..."
9,baseline_6_variables__shock_2019__eps_0.10,baseline_6_variables,main_candidate,2019,0.1000,"debt_capacity_score_0_100,employment_strength_...",6,,0,complete_case,True,True,"Balanced structural set across fiscal, labour,..."


In [6]:
# ------------------------------------------------------
# EPSILON SENSITIVITY RUNNER
# ------------------------------------------------------

def run_epsilon_sensitivity(run_config, snapshot):
    run_key = run_config["run_key"]
    set_name = run_config["set_name"]
    shock_id = str(run_config["shock_id"])
    epsilon = float(run_config["epsilon"])
    variables = parse_variable_list(run_config["variables"])

    missing_variables = [var for var in variables if var not in snapshot.columns]

    if missing_variables:
        raise ValueError(f"Run {run_key} has missing variables: {missing_variables}")

    run_df = snapshot[snapshot["shock_id"].astype(str) == shock_id].copy()

    if REQUIRE_RECOVERY_AVAILABLE and "Recovery" in run_df.columns:
        run_df = run_df[run_df["Recovery"].notna()].copy()

    run_df = run_df.dropna(subset=variables).copy()

    if run_df.empty:
        raise ValueError(f"Run {run_key} has zero rows after filtering.")

    scaled_cols = []

    for var in variables:
        scaled_col = f"{var}__unit"
        run_df[scaled_col] = min_max_scale_to_unit_interval(run_df[var])
        scaled_cols.append(scaled_col)

    run_df = run_df.dropna(subset=scaled_cols).copy()
    run_df = run_df.sort_values("Country").reset_index(drop=True)

    values = run_df[scaled_cols].to_numpy(dtype=float)

    if np.isclose(epsilon, 0.0):
        # At epsilon=0, use strict continuous dominance.
        dominance = build_strict_dominance_matrix(values)
        dominance_rule = "strict_continuous"
    else:
        dominance = build_epsilon_tolerance_dominance_matrix(values, epsilon)
        dominance_rule = "epsilon_tolerance"

    reciprocal_pairs = detect_reciprocal_pairs(dominance)
    is_dag = is_dag_kahn(dominance)
    is_valid_partial_order = is_dag and len(reciprocal_pairs) == 0

    layers = assign_layers_if_dag(dominance) if is_dag else None

    if layers is None:
        layers = {i: np.nan for i in range(len(run_df))}

    hasse_edges_idx = transitive_reduction_dag(dominance) if is_dag else []

    dominated_by_count = dominance.sum(axis=0)
    dominates_count = dominance.sum(axis=1)

    country_summary = run_df[
        ["Country", "country_name", "shock_id", "analysis_year", "Recovery"] + variables + scaled_cols
    ].copy()

    country_summary["dominates_country_count"] = dominates_count
    country_summary["dominated_by_country_count"] = dominated_by_count
    country_summary["is_pareto_frontier"] = country_summary["dominated_by_country_count"] == 0
    country_summary["epsilon_layer"] = country_summary.index.map(layers)

    dominance_rows = []

    for i in range(dominance.shape[0]):
        for j in range(dominance.shape[1]):
            if dominance[i, j]:
                dominance_rows.append({
                    "source_country": run_df.loc[i, "Country"],
                    "target_country": run_df.loc[j, "Country"],
                    "source_country_name": run_df.loc[i, "country_name"],
                    "target_country_name": run_df.loc[j, "country_name"],
                    "source_layer": layers.get(i, np.nan),
                    "target_layer": layers.get(j, np.nan),
                })

    dominance_relations = pd.DataFrame(dominance_rows)

    if dominance_relations.empty:
        dominance_relations = pd.DataFrame(columns=[
            "source_country",
            "target_country",
            "source_country_name",
            "target_country_name",
            "source_layer",
            "target_layer",
        ])

    hasse_edges = pd.DataFrame([
        {
            "source_country": run_df.loc[i, "Country"],
            "target_country": run_df.loc[j, "Country"],
            "source_country_name": run_df.loc[i, "country_name"],
            "target_country_name": run_df.loc[j, "country_name"],
            "source_layer": layers.get(i, np.nan),
            "target_layer": layers.get(j, np.nan),
        }
        for i, j in hasse_edges_idx
    ])

    if hasse_edges.empty:
        hasse_edges = pd.DataFrame(columns=[
            "source_country",
            "target_country",
            "source_country_name",
            "target_country_name",
            "source_layer",
            "target_layer",
        ])

    reciprocal_pair_table = pd.DataFrame([
        {
            "country_a": run_df.loc[i, "Country"],
            "country_b": run_df.loc[j, "Country"],
            "country_a_name": run_df.loc[i, "country_name"],
            "country_b_name": run_df.loc[j, "country_name"],
        }
        for i, j in reciprocal_pairs
    ])

    if reciprocal_pair_table.empty:
        reciprocal_pair_table = pd.DataFrame(columns=[
            "country_a",
            "country_b",
            "country_a_name",
            "country_b_name",
        ])

    pareto_countries = country_summary[country_summary["is_pareto_frontier"]].copy()

    layer_summary = (
        country_summary
        .groupby("epsilon_layer", dropna=False)
        .agg(
            country_count=("Country", "nunique"),
            countries=("Country", lambda x: ";".join(sorted(x.unique()))),
            recovery_median=("Recovery", "median"),
            recovery_mean=("Recovery", "mean"),
        )
        .reset_index()
    )

    metrics = comparability_metrics(dominance)

    run_summary = {
        "run_key": run_key,
        "set_name": set_name,
        "shock_id": shock_id,
        "epsilon": epsilon,
        "dominance_rule": dominance_rule,
        "sample_strategy": SAMPLE_STRATEGY,
        "require_recovery_available": REQUIRE_RECOVERY_AVAILABLE,
        "country_count": run_df["Country"].nunique(),
        "variable_count": len(variables),
        "variables": ",".join(variables),
        "dominance_relation_count": int(dominance.sum()),
        "hasse_edge_count": len(hasse_edges),
        "pareto_country_count": int(pareto_countries["Country"].nunique()),
        "pareto_country_set": ";".join(sorted(pareto_countries["Country"].unique())),
        "is_dag": bool(is_dag),
        "reciprocal_pair_count": len(reciprocal_pairs),
        "is_valid_partial_order": bool(is_valid_partial_order),
        "layer_count": int(pd.Series(list(layers.values())).dropna().nunique()) if layers else 0,
        **metrics,
        "is_working_main_set": bool(run_config["is_working_main_set"]),
    }

    metadata_cols = {
        "run_key": run_key,
        "set_name": set_name,
        "shock_id": shock_id,
        "epsilon": epsilon,
        "dominance_rule": dominance_rule,
        "is_valid_partial_order": bool(is_valid_partial_order),
    }

    tables = {
        "country_summary": country_summary,
        "dominance_relations": dominance_relations,
        "hasse_edges": hasse_edges,
        "pareto_countries": pareto_countries,
        "layer_summary": layer_summary,
        "reciprocal_pairs": reciprocal_pair_table,
    }

    for name, table in tables.items():
        for col, val in metadata_cols.items():
            if col in table.columns:
                table[col] = val
            else:
                table.insert(0, col, val)

        metadata_order = ["run_key", "set_name", "shock_id", "epsilon", "dominance_rule", "is_valid_partial_order"]
        ordered_cols = [col for col in metadata_order if col in table.columns]
        ordered_cols += [col for col in table.columns if col not in ordered_cols]
        tables[name] = table[ordered_cols].copy()

    return run_summary, tables

In [7]:
# ------------------------------------------------------
# EXECUTE EPSILON SENSITIVITY RUNS
# ------------------------------------------------------

run_summaries = []
run_errors = []

combined_tables = {
    "country_summary": [],
    "dominance_relations": [],
    "hasse_edges": [],
    "pareto_countries": [],
    "layer_summary": [],
    "reciprocal_pairs": [],
}

for _, run_config in epsilon_run_plan.iterrows():
    run_key = run_config["run_key"]

    if run_config["missing_variable_count"] > 0:
        run_errors.append({
            "run_key": run_key,
            "error": f"Missing variables: {run_config['missing_variables']}",
        })
        continue

    try:
        run_summary, tables = run_epsilon_sensitivity(run_config, snapshot)
        run_summaries.append(run_summary)

        for table_name, table in tables.items():
            combined_tables[table_name].append(table)

        print(f"Completed: {run_key}")

    except Exception as e:
        run_errors.append({
            "run_key": run_key,
            "error": str(e),
        })
        print(f"FAILED: {run_key} -> {e}")

epsilon_run_summary = pd.DataFrame(run_summaries)

expected_summary_cols = [
    "run_key", "set_name", "shock_id", "epsilon", "dominance_rule",
    "sample_strategy", "require_recovery_available", "country_count",
    "variable_count", "variables", "dominance_relation_count",
    "hasse_edge_count", "pareto_country_count", "pareto_country_set",
    "is_dag", "reciprocal_pair_count", "is_valid_partial_order",
    "layer_count", "comparable_pairs", "incomparable_pairs",
    "comparability_ratio", "incomparability_ratio", "is_working_main_set",
]

if epsilon_run_summary.empty:
    epsilon_run_summary = pd.DataFrame(columns=expected_summary_cols)

epsilon_run_errors = pd.DataFrame(run_errors)

if epsilon_run_errors.empty:
    epsilon_run_errors = pd.DataFrame(columns=["run_key", "error"])

epsilon_run_summary.to_csv(
    PROCESSED_DIR / "epsilon_run_summary.csv",
    index=False,
)

epsilon_run_summary.to_csv(
    DIAGNOSTICS_DIR / "epsilon_run_summary.csv",
    index=False,
)

epsilon_run_errors.to_csv(
    DIAGNOSTICS_DIR / "epsilon_run_errors.csv",
    index=False,
)

print("Completed runs:", len(epsilon_run_summary))
print("Failed runs:", len(epsilon_run_errors))

display(epsilon_run_summary)
display(epsilon_run_errors)

Completed: baseline_6_variables__shock_2007__eps_0.00
Completed: baseline_6_variables__shock_2007__eps_0.02
Completed: baseline_6_variables__shock_2007__eps_0.05
Completed: baseline_6_variables__shock_2007__eps_0.10
Completed: baseline_6_variables__shock_2007__eps_0.15
Completed: baseline_6_variables__shock_2007__eps_0.20
Completed: baseline_6_variables__shock_2019__eps_0.00
Completed: baseline_6_variables__shock_2019__eps_0.02
Completed: baseline_6_variables__shock_2019__eps_0.05
Completed: baseline_6_variables__shock_2019__eps_0.10
Completed: baseline_6_variables__shock_2019__eps_0.15
Completed: baseline_6_variables__shock_2019__eps_0.20
Completed: baseline_plus_gdp_stability_2007__shock_2007__eps_0.00
Completed: baseline_plus_gdp_stability_2007__shock_2007__eps_0.02
Completed: baseline_plus_gdp_stability_2007__shock_2007__eps_0.05
Completed: baseline_plus_gdp_stability_2007__shock_2007__eps_0.10
Completed: baseline_plus_gdp_stability_2007__shock_2007__eps_0.15
Completed: baseline_pl

,run_key,set_name,shock_id,epsilon,dominance_rule,sample_strategy,require_recovery_available,country_count,variable_count,variables,dominance_relation_count,hasse_edge_count,pareto_country_count,pareto_country_set,is_dag,reciprocal_pair_count,is_valid_partial_order,layer_count,comparable_pairs,incomparable_pairs,comparability_ratio,incomparability_ratio,is_working_main_set
0,baseline_6_variables__shock_2007__eps_0.00,baseline_6_variables,2007,0.0000,strict_continuous,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",46,40,13,BEL;CAN;CZE;DNK;EST;FIN;GBR;IRL;LTU;LUX;SVN;SW...,True,0,True,3,46,254,0.1533,0.8467,True
1,baseline_6_variables__shock_2007__eps_0.02,baseline_6_variables,2007,0.0200,epsilon_tolerance,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",52,45,13,BEL;CAN;CZE;DNK;EST;FIN;GBR;IRL;LTU;LUX;SVN;SW...,True,0,True,3,52,248,0.1733,0.8267,True
2,baseline_6_variables__shock_2007__eps_0.05,baseline_6_variables,2007,0.0500,epsilon_tolerance,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",64,54,10,CAN;DNK;EST;FIN;GBR;IRL;LUX;SVN;SWE;USA,True,0,True,3,64,236,0.2133,0.7867,True
3,baseline_6_variables__shock_2007__eps_0.10,baseline_6_variables,2007,0.1000,epsilon_tolerance,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",90,57,9,CAN;DNK;EST;FIN;GBR;IRL;LUX;SWE;USA,True,0,True,5,90,210,0.3000,0.7000,True
4,baseline_6_variables__shock_2007__eps_0.15,baseline_6_variables,2007,0.1500,epsilon_tolerance,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",113,63,7,CAN;DNK;EST;FIN;LUX;SWE;USA,True,0,True,5,113,187,0.3767,0.6233,True
5,baseline_6_variables__shock_2007__eps_0.20,baseline_6_variables,2007,0.2000,epsilon_tolerance,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",142,50,6,CAN;DNK;EST;FIN;SWE;USA,True,0,True,7,142,158,0.4733,0.5267,True
6,baseline_6_variables__shock_2019__eps_0.00,baseline_6_variables,2019,0.0000,strict_continuous,complete_case,True,32,6,"debt_capacity_score_0_100,employment_strength_...",60,50,21,AUS;AUT;BEL;CAN;CHE;CZE;DEU;DNK;EST;FIN;GBR;HU...,True,0,True,3,60,436,0.1210,0.8790,True
7,baseline_6_variables__shock_2019__eps_0.02,baseline_6_variables,2019,0.0200,epsilon_tolerance,complete_case,True,32,6,"debt_capacity_score_0_100,employment_strength_...",78,58,16,AUS;CAN;CHE;CZE;DEU;DNK;EST;FIN;GBR;IRL;KOR;LU...,True,0,True,4,78,418,0.1573,0.8427,True
8,baseline_6_variables__shock_2019__eps_0.05,baseline_6_variables,2019,0.0500,epsilon_tolerance,complete_case,True,32,6,"debt_capacity_score_0_100,employment_strength_...",111,71,12,AUS;CAN;CHE;CZE;DNK;EST;GBR;KOR;LUX;NZL;SWE;USA,True,0,True,4,111,385,0.2238,0.7762,True
9,baseline_6_variables__shock_2019__eps_0.10,baseline_6_variables,2019,0.1000,epsilon_tolerance,complete_case,True,32,6,"debt_capacity_score_0_100,employment_strength_...",178,59,10,AUS;CAN;CHE;EST;GBR;KOR;LUX;NZL;SWE;USA,True,0,True,6,178,318,0.3589,0.6411,True


,run_key,error


In [8]:
# ------------------------------------------------------
# EXPORT COMBINED EPSILON OUTPUTS
# ------------------------------------------------------

output_file_map = {
    "country_summary": "epsilon_country_summary.csv",
    "dominance_relations": "epsilon_dominance_relations.csv",
    "hasse_edges": "epsilon_hasse_edges.csv",
    "pareto_countries": "epsilon_pareto_countries.csv",
    "layer_summary": "epsilon_layer_summary.csv",
    "reciprocal_pairs": "epsilon_reciprocal_pairs.csv",
}

combined_output_tables = {}

for table_name, frames in combined_tables.items():
    if frames:
        combined = pd.concat(frames, ignore_index=True)
    else:
        combined = pd.DataFrame({"message": ["No rows produced for this output. Check epsilon_run_errors.csv if unexpected."]})

    combined_output_tables[table_name] = combined

    output_name = output_file_map[table_name]

    combined.to_csv(
        PROCESSED_DIR / output_name,
        index=False,
    )

    combined.to_csv(
        DIAGNOSTICS_DIR / output_name,
        index=False,
    )

    print(f"{output_name}: {len(combined)} rows")

epsilon_country_summary.csv: 1410 rows
epsilon_dominance_relations.csv: 6666 rows
epsilon_hasse_edges.csv: 2842 rows
epsilon_pareto_countries.csv: 527 rows
epsilon_layer_summary.csv: 283 rows
epsilon_reciprocal_pairs.csv: 0 rows


In [9]:
# ------------------------------------------------------
# CYCLE AND PARTIAL-ORDER DIAGNOSTICS
# ------------------------------------------------------

cycle_diagnostics = epsilon_run_summary[
    [
        "run_key",
        "set_name",
        "shock_id",
        "epsilon",
        "dominance_rule",
        "country_count",
        "dominance_relation_count",
        "is_dag",
        "reciprocal_pair_count",
        "is_valid_partial_order",
    ]
].copy()

cycle_diagnostics["diagnostic_status"] = np.where(
    cycle_diagnostics["is_valid_partial_order"],
    "valid_partial_order",
    np.where(
        cycle_diagnostics["reciprocal_pair_count"] > 0,
        "invalid_reciprocal_or_cycle_risk",
        "invalid_cycle_risk",
    ),
)

cycle_diagnostics.to_csv(
    PROCESSED_DIR / "epsilon_cycle_diagnostics.csv",
    index=False,
)

cycle_diagnostics.to_csv(
    DIAGNOSTICS_DIR / "epsilon_cycle_diagnostics.csv",
    index=False,
)

display(cycle_diagnostics)

,run_key,set_name,shock_id,epsilon,dominance_rule,country_count,dominance_relation_count,is_dag,reciprocal_pair_count,is_valid_partial_order,diagnostic_status
0,baseline_6_variables__shock_2007__eps_0.00,baseline_6_variables,2007,0.0000,strict_continuous,25,46,True,0,True,valid_partial_order
1,baseline_6_variables__shock_2007__eps_0.02,baseline_6_variables,2007,0.0200,epsilon_tolerance,25,52,True,0,True,valid_partial_order
2,baseline_6_variables__shock_2007__eps_0.05,baseline_6_variables,2007,0.0500,epsilon_tolerance,25,64,True,0,True,valid_partial_order
3,baseline_6_variables__shock_2007__eps_0.10,baseline_6_variables,2007,0.1000,epsilon_tolerance,25,90,True,0,True,valid_partial_order
4,baseline_6_variables__shock_2007__eps_0.15,baseline_6_variables,2007,0.1500,epsilon_tolerance,25,113,True,0,True,valid_partial_order
5,baseline_6_variables__shock_2007__eps_0.20,baseline_6_variables,2007,0.2000,epsilon_tolerance,25,142,True,0,True,valid_partial_order
6,baseline_6_variables__shock_2019__eps_0.00,baseline_6_variables,2019,0.0000,strict_continuous,32,60,True,0,True,valid_partial_order
7,baseline_6_variables__shock_2019__eps_0.02,baseline_6_variables,2019,0.0200,epsilon_tolerance,32,78,True,0,True,valid_partial_order
8,baseline_6_variables__shock_2019__eps_0.05,baseline_6_variables,2019,0.0500,epsilon_tolerance,32,111,True,0,True,valid_partial_order
9,baseline_6_variables__shock_2019__eps_0.10,baseline_6_variables,2019,0.1000,epsilon_tolerance,32,178,True,0,True,valid_partial_order


In [10]:
# ------------------------------------------------------
# FRONTIER STABILITY SUMMARY
# ------------------------------------------------------

frontier_rows = []

for (set_name, shock_id), group in epsilon_run_summary.groupby(["set_name", "shock_id"]):
    group = group.sort_values("epsilon").copy()

    base_rows = group[np.isclose(group["epsilon"], 0.0)]

    if base_rows.empty:
        continue

    base_frontier = set(str(base_rows.iloc[0]["pareto_country_set"]).split(";"))
    base_frontier = {country for country in base_frontier if country and country != "nan"}

    for _, row in group.iterrows():
        current_frontier = set(str(row["pareto_country_set"]).split(";"))
        current_frontier = {country for country in current_frontier if country and country != "nan"}

        frontier_rows.append({
            "set_name": set_name,
            "shock_id": shock_id,
            "epsilon": row["epsilon"],
            "dominance_rule": row["dominance_rule"],
            "is_valid_partial_order": row["is_valid_partial_order"],
            "pareto_country_count": row["pareto_country_count"],
            "pareto_country_set": row["pareto_country_set"],
            "baseline_epsilon_0_pareto_country_count": len(base_frontier),
            "jaccard_vs_epsilon_0": jaccard_similarity(base_frontier, current_frontier),
            "countries_entering_frontier_vs_epsilon_0": ";".join(sorted(current_frontier - base_frontier)),
            "countries_leaving_frontier_vs_epsilon_0": ";".join(sorted(base_frontier - current_frontier)),
        })

epsilon_frontier_stability_summary = pd.DataFrame(frontier_rows)

epsilon_frontier_stability_summary.to_csv(
    PROCESSED_DIR / "epsilon_frontier_stability_summary.csv",
    index=False,
)

epsilon_frontier_stability_summary.to_csv(
    DIAGNOSTICS_DIR / "epsilon_frontier_stability_summary.csv",
    index=False,
)

display(epsilon_frontier_stability_summary)

,set_name,shock_id,epsilon,dominance_rule,is_valid_partial_order,pareto_country_count,pareto_country_set,baseline_epsilon_0_pareto_country_count,jaccard_vs_epsilon_0,countries_entering_frontier_vs_epsilon_0,countries_leaving_frontier_vs_epsilon_0
0,baseline_6_variables,2007,0.0000,strict_continuous,True,13,BEL;CAN;CZE;DNK;EST;FIN;GBR;IRL;LTU;LUX;SVN;SW...,13,1.0000,,
1,baseline_6_variables,2007,0.0200,epsilon_tolerance,True,13,BEL;CAN;CZE;DNK;EST;FIN;GBR;IRL;LTU;LUX;SVN;SW...,13,1.0000,,
2,baseline_6_variables,2007,0.0500,epsilon_tolerance,True,10,CAN;DNK;EST;FIN;GBR;IRL;LUX;SVN;SWE;USA,13,0.7692,,BEL;CZE;LTU
3,baseline_6_variables,2007,0.1000,epsilon_tolerance,True,9,CAN;DNK;EST;FIN;GBR;IRL;LUX;SWE;USA,13,0.6923,,BEL;CZE;LTU;SVN
4,baseline_6_variables,2007,0.1500,epsilon_tolerance,True,7,CAN;DNK;EST;FIN;LUX;SWE;USA,13,0.5385,,BEL;CZE;GBR;IRL;LTU;SVN
5,baseline_6_variables,2007,0.2000,epsilon_tolerance,True,6,CAN;DNK;EST;FIN;SWE;USA,13,0.4615,,BEL;CZE;GBR;IRL;LTU;LUX;SVN
6,baseline_6_variables,2019,0.0000,strict_continuous,True,21,AUS;AUT;BEL;CAN;CHE;CZE;DEU;DNK;EST;FIN;GBR;HU...,21,1.0000,,
7,baseline_6_variables,2019,0.0200,epsilon_tolerance,True,16,AUS;CAN;CHE;CZE;DEU;DNK;EST;FIN;GBR;IRL;KOR;LU...,21,0.7619,,AUT;BEL;HUN;LTU;SVN
8,baseline_6_variables,2019,0.0500,epsilon_tolerance,True,12,AUS;CAN;CHE;CZE;DNK;EST;GBR;KOR;LUX;NZL;SWE;USA,21,0.5714,,AUT;BEL;DEU;FIN;HUN;IRL;LTU;POL;SVN
9,baseline_6_variables,2019,0.1000,epsilon_tolerance,True,10,AUS;CAN;CHE;EST;GBR;KOR;LUX;NZL;SWE;USA,21,0.4762,,AUT;BEL;CZE;DEU;DNK;FIN;HUN;IRL;LTU;POL;SVN


In [11]:
# ------------------------------------------------------
# WORKING MAIN EPSILON OUTPUTS
# ------------------------------------------------------

working_main_epsilon_summary = epsilon_run_summary[
    epsilon_run_summary["set_name"] == MAIN_SET_NAME
].copy()

working_main_frontier_stability = epsilon_frontier_stability_summary[
    epsilon_frontier_stability_summary["set_name"] == MAIN_SET_NAME
].copy()

working_main_country_summary = combined_output_tables["country_summary"]
working_main_pareto_countries = combined_output_tables["pareto_countries"]

if "set_name" in working_main_country_summary.columns:
    working_main_country_summary = working_main_country_summary[
        working_main_country_summary["set_name"] == MAIN_SET_NAME
    ].copy()

if "set_name" in working_main_pareto_countries.columns:
    working_main_pareto_countries = working_main_pareto_countries[
        working_main_pareto_countries["set_name"] == MAIN_SET_NAME
    ].copy()

working_main_epsilon_summary.to_csv(
    PROCESSED_DIR / "working_main_epsilon_run_summary.csv",
    index=False,
)

working_main_frontier_stability.to_csv(
    PROCESSED_DIR / "working_main_epsilon_frontier_stability_summary.csv",
    index=False,
)

working_main_country_summary.to_csv(
    PROCESSED_DIR / "working_main_epsilon_country_summary.csv",
    index=False,
)

working_main_pareto_countries.to_csv(
    PROCESSED_DIR / "working_main_epsilon_pareto_countries.csv",
    index=False,
)

display(working_main_epsilon_summary)
display(working_main_frontier_stability)
display(working_main_pareto_countries.head(100))

,run_key,set_name,shock_id,epsilon,dominance_rule,sample_strategy,require_recovery_available,country_count,variable_count,variables,dominance_relation_count,hasse_edge_count,pareto_country_count,pareto_country_set,is_dag,reciprocal_pair_count,is_valid_partial_order,layer_count,comparable_pairs,incomparable_pairs,comparability_ratio,incomparability_ratio,is_working_main_set
0,baseline_6_variables__shock_2007__eps_0.00,baseline_6_variables,2007,0.0000,strict_continuous,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",46,40,13,BEL;CAN;CZE;DNK;EST;FIN;GBR;IRL;LTU;LUX;SVN;SW...,True,0,True,3,46,254,0.1533,0.8467,True
1,baseline_6_variables__shock_2007__eps_0.02,baseline_6_variables,2007,0.0200,epsilon_tolerance,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",52,45,13,BEL;CAN;CZE;DNK;EST;FIN;GBR;IRL;LTU;LUX;SVN;SW...,True,0,True,3,52,248,0.1733,0.8267,True
2,baseline_6_variables__shock_2007__eps_0.05,baseline_6_variables,2007,0.0500,epsilon_tolerance,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",64,54,10,CAN;DNK;EST;FIN;GBR;IRL;LUX;SVN;SWE;USA,True,0,True,3,64,236,0.2133,0.7867,True
3,baseline_6_variables__shock_2007__eps_0.10,baseline_6_variables,2007,0.1000,epsilon_tolerance,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",90,57,9,CAN;DNK;EST;FIN;GBR;IRL;LUX;SWE;USA,True,0,True,5,90,210,0.3000,0.7000,True
4,baseline_6_variables__shock_2007__eps_0.15,baseline_6_variables,2007,0.1500,epsilon_tolerance,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",113,63,7,CAN;DNK;EST;FIN;LUX;SWE;USA,True,0,True,5,113,187,0.3767,0.6233,True
5,baseline_6_variables__shock_2007__eps_0.20,baseline_6_variables,2007,0.2000,epsilon_tolerance,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",142,50,6,CAN;DNK;EST;FIN;SWE;USA,True,0,True,7,142,158,0.4733,0.5267,True
6,baseline_6_variables__shock_2019__eps_0.00,baseline_6_variables,2019,0.0000,strict_continuous,complete_case,True,32,6,"debt_capacity_score_0_100,employment_strength_...",60,50,21,AUS;AUT;BEL;CAN;CHE;CZE;DEU;DNK;EST;FIN;GBR;HU...,True,0,True,3,60,436,0.1210,0.8790,True
7,baseline_6_variables__shock_2019__eps_0.02,baseline_6_variables,2019,0.0200,epsilon_tolerance,complete_case,True,32,6,"debt_capacity_score_0_100,employment_strength_...",78,58,16,AUS;CAN;CHE;CZE;DEU;DNK;EST;FIN;GBR;IRL;KOR;LU...,True,0,True,4,78,418,0.1573,0.8427,True
8,baseline_6_variables__shock_2019__eps_0.05,baseline_6_variables,2019,0.0500,epsilon_tolerance,complete_case,True,32,6,"debt_capacity_score_0_100,employment_strength_...",111,71,12,AUS;CAN;CHE;CZE;DNK;EST;GBR;KOR;LUX;NZL;SWE;USA,True,0,True,4,111,385,0.2238,0.7762,True
9,baseline_6_variables__shock_2019__eps_0.10,baseline_6_variables,2019,0.1000,epsilon_tolerance,complete_case,True,32,6,"debt_capacity_score_0_100,employment_strength_...",178,59,10,AUS;CAN;CHE;EST;GBR;KOR;LUX;NZL;SWE;USA,True,0,True,6,178,318,0.3589,0.6411,True


,set_name,shock_id,epsilon,dominance_rule,is_valid_partial_order,pareto_country_count,pareto_country_set,baseline_epsilon_0_pareto_country_count,jaccard_vs_epsilon_0,countries_entering_frontier_vs_epsilon_0,countries_leaving_frontier_vs_epsilon_0
0,baseline_6_variables,2007,0.0000,strict_continuous,True,13,BEL;CAN;CZE;DNK;EST;FIN;GBR;IRL;LTU;LUX;SVN;SW...,13,1.0000,,
1,baseline_6_variables,2007,0.0200,epsilon_tolerance,True,13,BEL;CAN;CZE;DNK;EST;FIN;GBR;IRL;LTU;LUX;SVN;SW...,13,1.0000,,
2,baseline_6_variables,2007,0.0500,epsilon_tolerance,True,10,CAN;DNK;EST;FIN;GBR;IRL;LUX;SVN;SWE;USA,13,0.7692,,BEL;CZE;LTU
3,baseline_6_variables,2007,0.1000,epsilon_tolerance,True,9,CAN;DNK;EST;FIN;GBR;IRL;LUX;SWE;USA,13,0.6923,,BEL;CZE;LTU;SVN
4,baseline_6_variables,2007,0.1500,epsilon_tolerance,True,7,CAN;DNK;EST;FIN;LUX;SWE;USA,13,0.5385,,BEL;CZE;GBR;IRL;LTU;SVN
5,baseline_6_variables,2007,0.2000,epsilon_tolerance,True,6,CAN;DNK;EST;FIN;SWE;USA,13,0.4615,,BEL;CZE;GBR;IRL;LTU;LUX;SVN
6,baseline_6_variables,2019,0.0000,strict_continuous,True,21,AUS;AUT;BEL;CAN;CHE;CZE;DEU;DNK;EST;FIN;GBR;HU...,21,1.0000,,
7,baseline_6_variables,2019,0.0200,epsilon_tolerance,True,16,AUS;CAN;CHE;CZE;DEU;DNK;EST;FIN;GBR;IRL;KOR;LU...,21,0.7619,,AUT;BEL;HUN;LTU;SVN
8,baseline_6_variables,2019,0.0500,epsilon_tolerance,True,12,AUS;CAN;CHE;CZE;DNK;EST;GBR;KOR;LUX;NZL;SWE;USA,21,0.5714,,AUT;BEL;DEU;FIN;HUN;IRL;LTU;POL;SVN
9,baseline_6_variables,2019,0.1000,epsilon_tolerance,True,10,AUS;CAN;CHE;EST;GBR;KOR;LUX;NZL;SWE;USA,21,0.4762,,AUT;BEL;CZE;DEU;DNK;FIN;HUN;IRL;LTU;POL;SVN


,run_key,set_name,shock_id,epsilon,dominance_rule,is_valid_partial_order,Country,country_name,analysis_year,Recovery,debt_capacity_score_0_100,employment_strength_score_0_100,rd_intensity_score_0_100,human_capital_tertiary_score_0_100,energy_security_score_0_100,governance_capacity_score_0_100,debt_capacity_score_0_100__unit,employment_strength_score_0_100__unit,rd_intensity_score_0_100__unit,human_capital_tertiary_score_0_100__unit,energy_security_score_0_100__unit,governance_capacity_score_0_100__unit,dominates_country_count,dominated_by_country_count,is_pareto_frontier,epsilon_layer,gdp_growth_stability_pre_2007_score_0_100,gdp_growth_stability_pre_2007_score_0_100__unit,gdp_growth_stability_pre_2019_score_0_100,gdp_growth_stability_pre_2019_score_0_100__unit,productivity_capacity_score_0_100,productivity_capacity_score_0_100__unit
0,baseline_6_variables__shock_2007__eps_0.00,baseline_6_variables,2007,0.0000,strict_continuous,True,BEL,Belgium,2007,1.0000,26.4183,84.6671,40.5810,61.0357,11.5940,90.3204,0.1698,0.4658,0.4854,0.5351,0.0961,0.6832,0,0,True,1,NaN,NaN,NaN,NaN,NaN,NaN
1,baseline_6_variables__shock_2007__eps_0.00,baseline_6_variables,2007,0.0000,strict_continuous,True,CAN,Canada,2007,1.0000,68.9667,88.6913,41.8854,100.0000,20.9717,88.1037,0.6499,0.6367,0.5039,1.0000,1.0000,0.6061,4,0,True,1,NaN,NaN,NaN,NaN,NaN,NaN
2,baseline_6_variables__shock_2007__eps_0.00,baseline_6_variables,2007,0.0000,strict_continuous,True,CZE,Czechia,2007,5.0000,79.4042,91.2674,27.1111,16.5599,15.8418,78.2822,0.7676,0.7461,0.2939,0.0045,0.5056,0.2648,1,0,True,1,NaN,NaN,NaN,NaN,NaN,NaN
3,baseline_6_variables__shock_2007__eps_0.00,baseline_6_variables,2007,0.0000,strict_continuous,True,DNK,Denmark,2007,7.0000,77.4678,97.2452,56.6294,58.1500,19.1805,96.3165,0.7458,1.0000,0.7135,0.5007,0.8274,0.8915,11,0,True,1,NaN,NaN,NaN,NaN,NaN,NaN
4,baseline_6_variables__shock_2007__eps_0.00,baseline_6_variables,2007,0.0000,strict_continuous,True,EST,Estonia,2007,8.0000,100.0000,93.5291,21.3168,63.5817,15.8072,92.9690,1.0000,0.8422,0.2115,0.5655,0.5022,0.7752,5,0,True,1,NaN,NaN,NaN,NaN,NaN,NaN
5,baseline_6_variables__shock_2007__eps_0.00,baseline_6_variables,2007,0.0000,strict_continuous,True,FIN,Finland,2007,8.0000,71.7468,86.6114,76.7884,71.3594,13.9414,86.6749,0.6812,0.5484,1.0000,0.6583,0.3224,0.5565,3,0,True,1,NaN,NaN,NaN,NaN,NaN,NaN
6,baseline_6_variables__shock_2007__eps_0.00,baseline_6_variables,2007,0.0000,strict_continuous,True,GBR,United Kingdom,2007,5.0000,21.2486,91.7511,34.7603,69.3707,16.0547,96.9669,0.1115,0.7667,0.4026,0.6346,0.5261,0.9141,2,0,True,1,NaN,NaN,NaN,NaN,NaN,NaN
7,baseline_6_variables__shock_2007__eps_0.00,baseline_6_variables,2007,0.0000,strict_continuous,True,IRL,Ireland,2007,6.0000,82.3967,92.8666,25.5868,61.7509,11.3965,99.4377,0.8014,0.8140,0.2722,0.5436,0.0771,1.0000,0,0,True,1,NaN,NaN,NaN,NaN,NaN,NaN
8,baseline_6_variables__shock_2007__eps_0.00,baseline_6_variables,2007,0.0000,strict_continuous,True,LTU,Lithuania,2007,5.0000,89.4380,94.5521,15.0461,53.3734,13.5295,73.2800,0.8808,0.8856,0.1224,0.4437,0.2827,0.0910,2,0,True,1,NaN,NaN,NaN,NaN,NaN,NaN
9,baseline_6_variables__shock_2007__eps_0.00,baseline_6_variables,2007,0.0000,strict_continuous,True,LUX,Luxembourg,2007,2.0000,96.3033,95.1222,33.8048,47.5149,10.5965,96.5962,0.9583,0.9098,0.3890,0.3738,0.0000,0.9013,0,0,True,1,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
# ------------------------------------------------------
# EPSILON SUMMARY FIGURES
# ------------------------------------------------------

if not working_main_epsilon_summary.empty:
    for shock_id, group in working_main_epsilon_summary.groupby("shock_id"):
        group = group.sort_values("epsilon")

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(group["epsilon"], group["comparability_ratio"], marker="o")
        ax.set_title(f"Comparability ratio vs epsilon — {MAIN_SET_NAME}, shock {shock_id}")
        ax.set_xlabel("Epsilon")
        ax.set_ylabel("Comparability ratio")
        ax.grid(alpha=0.25)
        fig.tight_layout()

        path = FIGURES_DIR / f"working_main_comparability_vs_epsilon_shock_{shock_id}.png"
        fig.savefig(path, dpi=200, bbox_inches="tight")
        plt.close(fig)

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(group["epsilon"], group["pareto_country_count"], marker="o")
        ax.set_title(f"Pareto country count vs epsilon — {MAIN_SET_NAME}, shock {shock_id}")
        ax.set_xlabel("Epsilon")
        ax.set_ylabel("Pareto country count")
        ax.grid(alpha=0.25)
        fig.tight_layout()

        path = FIGURES_DIR / f"working_main_pareto_count_vs_epsilon_shock_{shock_id}.png"
        fig.savefig(path, dpi=200, bbox_inches="tight")
        plt.close(fig)

print("Figures created in:")
print(FIGURES_DIR.resolve())

Figures created in:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Outputs\Figures\Epsilon_Sensitivity_Country_Level


In [13]:
# ------------------------------------------------------
# DATA DICTIONARY AND METHODOLOGICAL NOTES
# ------------------------------------------------------

epsilon_data_dictionary = pd.DataFrame([
    {
        "table": "epsilon_run_summary.csv",
        "column": "epsilon",
        "description": "Tolerance value applied after variables are scaled to [0, 1] within each run.",
    },
    {
        "table": "epsilon_run_summary.csv",
        "column": "dominance_rule",
        "description": "strict_continuous for epsilon 0, epsilon_tolerance for epsilon > 0.",
    },
    {
        "table": "epsilon_run_summary.csv",
        "column": "is_valid_partial_order",
        "description": "True if the epsilon relation is acyclic and has no reciprocal dominance pairs.",
    },
    {
        "table": "epsilon_run_summary.csv",
        "column": "comparability_ratio",
        "description": "Share of country pairs comparable under the epsilon dominance relation.",
    },
    {
        "table": "epsilon_country_summary.csv",
        "column": "is_pareto_frontier",
        "description": "True if the country is not dominated under the current epsilon relation.",
    },
    {
        "table": "epsilon_frontier_stability_summary.csv",
        "column": "jaccard_vs_epsilon_0",
        "description": "Similarity of the Pareto frontier to the epsilon=0 frontier.",
    },
    {
        "table": "epsilon_reciprocal_pairs.csv",
        "column": "country_a/country_b",
        "description": "Country pairs that dominate each other under the tolerance rule, indicating invalid POSet behavior.",
    },
])

epsilon_data_dictionary.to_csv(
    PROCESSED_DIR / "epsilon_data_dictionary.csv",
    index=False,
)

methodological_notes = pd.DataFrame([
    {
        "topic": "Continuous variables",
        "note": "This notebook uses continuous direction-aligned variables scaled to [0, 1], not discretized profile codes.",
    },
    {
        "topic": "Epsilon tolerance rule",
        "note": "For epsilon > 0, A epsilon-dominates B if A + epsilon >= B in every dimension and A > B + epsilon in at least one dimension.",
    },
    {
        "topic": "Cycle risk",
        "note": "The tolerance rule can create cycles or reciprocal dominance, so every run is checked for partial-order validity.",
    },
    {
        "topic": "Report caution",
        "note": "Invalid epsilon runs should not be presented as valid POSet/Hasse structures. They are sensitivity diagnostics.",
    },
    {
        "topic": "Recovery",
        "note": "Recovery is retained only for later validation and is never used in the dominance relation.",
    },
    {
        "topic": "Next step",
        "note": "The safer epsilon-margin robustness rule is implemented in the next notebook, 09_Epsilon_Margin_POSet_Robustness.ipynb.",
    },
])

methodological_notes.to_csv(
    DIAGNOSTICS_DIR / "epsilon_methodological_notes.csv",
    index=False,
)

display(epsilon_data_dictionary)
display(methodological_notes)

,table,column,description
0,epsilon_run_summary.csv,epsilon,Tolerance value applied after variables are sc...
1,epsilon_run_summary.csv,dominance_rule,"strict_continuous for epsilon 0, epsilon_toler..."
2,epsilon_run_summary.csv,is_valid_partial_order,True if the epsilon relation is acyclic and ha...
3,epsilon_run_summary.csv,comparability_ratio,Share of country pairs comparable under the ep...
4,epsilon_country_summary.csv,is_pareto_frontier,True if the country is not dominated under the...
5,epsilon_frontier_stability_summary.csv,jaccard_vs_epsilon_0,Similarity of the Pareto frontier to the epsil...
6,epsilon_reciprocal_pairs.csv,country_a/country_b,Country pairs that dominate each other under t...


,topic,note
0,Continuous variables,This notebook uses continuous direction-aligne...
1,Epsilon tolerance rule,"For epsilon > 0, A epsilon-dominates B if A + ..."
2,Cycle risk,The tolerance rule can create cycles or recipr...
3,Report caution,Invalid epsilon runs should not be presented a...
4,Recovery,Recovery is retained only for later validation...
5,Next step,The safer epsilon-margin robustness rule is im...


In [14]:
# ------------------------------------------------------
# CREATE EPSILON AUDIT WORKBOOK
# ------------------------------------------------------

EPSILON_AUDIT_XLSX = AUDIT_DIR / "08_Epsilon_Sensitivity_Country_Level_Audit.xlsx"

audit_sources = [
    {
        "sheet_name": "01_run_plan",
        "description": "All configured epsilon runs.",
        "path": PROCESSED_DIR / "epsilon_run_plan.csv",
    },
    {
        "sheet_name": "02_run_summary",
        "description": "Summary metrics for all epsilon runs.",
        "path": PROCESSED_DIR / "epsilon_run_summary.csv",
    },
    {
        "sheet_name": "03_run_errors",
        "description": "Errors encountered while running epsilon configurations.",
        "path": DIAGNOSTICS_DIR / "epsilon_run_errors.csv",
    },
    {
        "sheet_name": "04_cycle_diag",
        "description": "Cycle and partial-order validity diagnostics.",
        "path": PROCESSED_DIR / "epsilon_cycle_diagnostics.csv",
    },
    {
        "sheet_name": "05_frontier_stability",
        "description": "Pareto frontier stability across epsilon values.",
        "path": PROCESSED_DIR / "epsilon_frontier_stability_summary.csv",
    },
    {
        "sheet_name": "06_working_main",
        "description": "Working main epsilon summary.",
        "path": PROCESSED_DIR / "working_main_epsilon_run_summary.csv",
    },
    {
        "sheet_name": "07_working_frontier",
        "description": "Working main frontier stability.",
        "path": PROCESSED_DIR / "working_main_epsilon_frontier_stability_summary.csv",
    },
    {
        "sheet_name": "08_pareto_countries",
        "description": "Pareto countries across all epsilon runs.",
        "path": PROCESSED_DIR / "epsilon_pareto_countries.csv",
    },
    {
        "sheet_name": "09_reciprocal_pairs",
        "description": "Reciprocal dominance pairs.",
        "path": PROCESSED_DIR / "epsilon_reciprocal_pairs.csv",
    },
    {
        "sheet_name": "10_dictionary",
        "description": "Epsilon output data dictionary.",
        "path": PROCESSED_DIR / "epsilon_data_dictionary.csv",
    },
    {
        "sheet_name": "11_method_notes",
        "description": "Methodological notes.",
        "path": DIAGNOSTICS_DIR / "epsilon_methodological_notes.csv",
    },
]

used_sheets = set()

with pd.ExcelWriter(EPSILON_AUDIT_XLSX, engine="xlsxwriter") as writer:
    workbook = writer.book

    title_fmt = workbook.add_format({
        "bold": True,
        "font_size": 15,
        "font_color": "white",
        "bg_color": "#1F4E78",
        "align": "left",
        "valign": "vcenter",
    })

    desc_fmt = workbook.add_format({
        "text_wrap": True,
        "font_size": 10,
        "font_color": "#333333",
        "valign": "top",
    })

    header_fmt = workbook.add_format({
        "bold": True,
        "font_color": "white",
        "bg_color": "#5B9BD5",
        "border": 1,
        "align": "center",
        "valign": "vcenter",
        "text_wrap": True,
    })

    index_rows = []

    for item in audit_sources:
        path = Path(item["path"])

        if path.exists():
            try:
                df_temp = pd.read_csv(path)
                rows = len(df_temp)
                cols = len(df_temp.columns)
            except Exception:
                rows = np.nan
                cols = np.nan
        else:
            rows = 0
            cols = 0

        index_rows.append({
            "Sheet": item["sheet_name"],
            "Rows": rows,
            "Columns": cols,
            "Description": item["description"],
            "Path": str(path),
        })

    index_df = pd.DataFrame(index_rows)
    index_df.to_excel(writer, sheet_name="00_INDEX", index=False, startrow=5)

    ws = writer.sheets["00_INDEX"]
    ws.merge_range(0, 0, 0, max(4, len(index_df.columns) - 1), "08 Epsilon Sensitivity Country-Level Audit", title_fmt)
    ws.merge_range(1, 0, 3, max(4, len(index_df.columns) - 1), "Audit workbook for country-level epsilon sensitivity diagnostics.", desc_fmt)

    for col_idx, col_name in enumerate(index_df.columns):
        ws.write(5, col_idx, col_name, header_fmt)
        ws.set_column(col_idx, col_idx, estimate_width(index_df[col_name], col_name))

    ws.autofilter(5, 0, 5 + len(index_df), len(index_df.columns) - 1)
    ws.freeze_panes(6, 0)

    for item in audit_sources:
        path = Path(item["path"])

        if path.exists():
            try:
                df_sheet = pd.read_csv(path)
            except Exception as e:
                df_sheet = pd.DataFrame({"message": [f"Could not read file: {e}"]})
        else:
            df_sheet = pd.DataFrame({"message": ["File not found."]})

        if df_sheet.empty:
            df_sheet = pd.DataFrame({"message": ["No rows in this table."]})

        sheet_name = safe_sheet_name(item["sheet_name"], used_sheets)

        df_sheet.to_excel(writer, sheet_name=sheet_name, index=False, startrow=4)

        ws = writer.sheets[sheet_name]
        max_col = max(4, len(df_sheet.columns) - 1)

        ws.merge_range(0, 0, 0, max_col, item["sheet_name"], title_fmt)
        ws.merge_range(1, 0, 3, max_col, item["description"], desc_fmt)

        for col_idx, col_name in enumerate(df_sheet.columns):
            ws.write(4, col_idx, col_name, header_fmt)
            ws.set_column(col_idx, col_idx, estimate_width(df_sheet[col_name], col_name))

        ws.autofilter(4, 0, 4 + len(df_sheet), len(df_sheet.columns) - 1)
        ws.freeze_panes(5, 0)

print("Epsilon audit workbook created:")
print(EPSILON_AUDIT_XLSX.resolve())

Epsilon audit workbook created:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Audit\08_Epsilon_Sensitivity_Country_Level_Audit.xlsx


In [15]:
# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

print("08 EPSILON SENSITIVITY COUNTRY-LEVEL COMPLETE")
print("=" * 80)

print("\nProcessed folder:")
print(PROCESSED_DIR.resolve())

print("\nDiagnostics folder:")
print(DIAGNOSTICS_DIR.resolve())

print("\nFigures folder:")
print(FIGURES_DIR.resolve())

print("\nAudit workbook:")
print(EPSILON_AUDIT_XLSX.resolve())

print("\nMain processed outputs:")
main_outputs = [
    "epsilon_run_plan.csv",
    "epsilon_run_summary.csv",
    "epsilon_country_summary.csv",
    "epsilon_dominance_relations.csv",
    "epsilon_hasse_edges.csv",
    "epsilon_pareto_countries.csv",
    "epsilon_cycle_diagnostics.csv",
    "epsilon_frontier_stability_summary.csv",
    "working_main_epsilon_run_summary.csv",
    "working_main_epsilon_frontier_stability_summary.csv",
]

for file_name in main_outputs:
    path = PROCESSED_DIR / file_name
    status = "FOUND" if path.exists() else "MISSING"
    print(f"- {status}: {file_name}")

print("\nWorking main summary:")
display(working_main_epsilon_summary)

print("\nCycle diagnostics:")
display(cycle_diagnostics)

print("\nImportant notes:")
print("1. Recovery was not used as an ordering variable.")
print("2. Epsilon > 0 uses a diagnostic tolerance rule and may create cycles.")
print("3. Invalid partial-order runs should not be used as Hasse/POSet conclusions.")
print("4. The safer epsilon-margin robustness check comes next.")

print("\nNext notebook:")
print("09_Epsilon_Margin_POSet_Robustness.ipynb")

08 EPSILON SENSITIVITY COUNTRY-LEVEL COMPLETE

Processed folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Epsilon_Sensitivity_Country_Level

Diagnostics folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Diagnostics\08_Epsilon_Sensitivity_Country_Level

Figures folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Outputs\Figures\Epsilon_Sensitivity_Country_Level

Audit workbook:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Audit\08_Epsilon_Sensitivity_Country_Level_Audit.xlsx

Main processed outputs:
- FOUND: epsilon_run_plan.csv
- FOUND: epsilon_run_summary.csv
- FOUND: epsilon_country_summary.csv
- FOUND: epsilon_dominance_relations.csv
- FOUND: epsilon_hasse_edges.csv
- FOUND: epsilon_

,run_key,set_name,shock_id,epsilon,dominance_rule,sample_strategy,require_recovery_available,country_count,variable_count,variables,dominance_relation_count,hasse_edge_count,pareto_country_count,pareto_country_set,is_dag,reciprocal_pair_count,is_valid_partial_order,layer_count,comparable_pairs,incomparable_pairs,comparability_ratio,incomparability_ratio,is_working_main_set
0,baseline_6_variables__shock_2007__eps_0.00,baseline_6_variables,2007,0.0000,strict_continuous,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",46,40,13,BEL;CAN;CZE;DNK;EST;FIN;GBR;IRL;LTU;LUX;SVN;SW...,True,0,True,3,46,254,0.1533,0.8467,True
1,baseline_6_variables__shock_2007__eps_0.02,baseline_6_variables,2007,0.0200,epsilon_tolerance,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",52,45,13,BEL;CAN;CZE;DNK;EST;FIN;GBR;IRL;LTU;LUX;SVN;SW...,True,0,True,3,52,248,0.1733,0.8267,True
2,baseline_6_variables__shock_2007__eps_0.05,baseline_6_variables,2007,0.0500,epsilon_tolerance,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",64,54,10,CAN;DNK;EST;FIN;GBR;IRL;LUX;SVN;SWE;USA,True,0,True,3,64,236,0.2133,0.7867,True
3,baseline_6_variables__shock_2007__eps_0.10,baseline_6_variables,2007,0.1000,epsilon_tolerance,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",90,57,9,CAN;DNK;EST;FIN;GBR;IRL;LUX;SWE;USA,True,0,True,5,90,210,0.3000,0.7000,True
4,baseline_6_variables__shock_2007__eps_0.15,baseline_6_variables,2007,0.1500,epsilon_tolerance,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",113,63,7,CAN;DNK;EST;FIN;LUX;SWE;USA,True,0,True,5,113,187,0.3767,0.6233,True
5,baseline_6_variables__shock_2007__eps_0.20,baseline_6_variables,2007,0.2000,epsilon_tolerance,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",142,50,6,CAN;DNK;EST;FIN;SWE;USA,True,0,True,7,142,158,0.4733,0.5267,True
6,baseline_6_variables__shock_2019__eps_0.00,baseline_6_variables,2019,0.0000,strict_continuous,complete_case,True,32,6,"debt_capacity_score_0_100,employment_strength_...",60,50,21,AUS;AUT;BEL;CAN;CHE;CZE;DEU;DNK;EST;FIN;GBR;HU...,True,0,True,3,60,436,0.1210,0.8790,True
7,baseline_6_variables__shock_2019__eps_0.02,baseline_6_variables,2019,0.0200,epsilon_tolerance,complete_case,True,32,6,"debt_capacity_score_0_100,employment_strength_...",78,58,16,AUS;CAN;CHE;CZE;DEU;DNK;EST;FIN;GBR;IRL;KOR;LU...,True,0,True,4,78,418,0.1573,0.8427,True
8,baseline_6_variables__shock_2019__eps_0.05,baseline_6_variables,2019,0.0500,epsilon_tolerance,complete_case,True,32,6,"debt_capacity_score_0_100,employment_strength_...",111,71,12,AUS;CAN;CHE;CZE;DNK;EST;GBR;KOR;LUX;NZL;SWE;USA,True,0,True,4,111,385,0.2238,0.7762,True
9,baseline_6_variables__shock_2019__eps_0.10,baseline_6_variables,2019,0.1000,epsilon_tolerance,complete_case,True,32,6,"debt_capacity_score_0_100,employment_strength_...",178,59,10,AUS;CAN;CHE;EST;GBR;KOR;LUX;NZL;SWE;USA,True,0,True,6,178,318,0.3589,0.6411,True



Cycle diagnostics:


,run_key,set_name,shock_id,epsilon,dominance_rule,country_count,dominance_relation_count,is_dag,reciprocal_pair_count,is_valid_partial_order,diagnostic_status
0,baseline_6_variables__shock_2007__eps_0.00,baseline_6_variables,2007,0.0000,strict_continuous,25,46,True,0,True,valid_partial_order
1,baseline_6_variables__shock_2007__eps_0.02,baseline_6_variables,2007,0.0200,epsilon_tolerance,25,52,True,0,True,valid_partial_order
2,baseline_6_variables__shock_2007__eps_0.05,baseline_6_variables,2007,0.0500,epsilon_tolerance,25,64,True,0,True,valid_partial_order
3,baseline_6_variables__shock_2007__eps_0.10,baseline_6_variables,2007,0.1000,epsilon_tolerance,25,90,True,0,True,valid_partial_order
4,baseline_6_variables__shock_2007__eps_0.15,baseline_6_variables,2007,0.1500,epsilon_tolerance,25,113,True,0,True,valid_partial_order
5,baseline_6_variables__shock_2007__eps_0.20,baseline_6_variables,2007,0.2000,epsilon_tolerance,25,142,True,0,True,valid_partial_order
6,baseline_6_variables__shock_2019__eps_0.00,baseline_6_variables,2019,0.0000,strict_continuous,32,60,True,0,True,valid_partial_order
7,baseline_6_variables__shock_2019__eps_0.02,baseline_6_variables,2019,0.0200,epsilon_tolerance,32,78,True,0,True,valid_partial_order
8,baseline_6_variables__shock_2019__eps_0.05,baseline_6_variables,2019,0.0500,epsilon_tolerance,32,111,True,0,True,valid_partial_order
9,baseline_6_variables__shock_2019__eps_0.10,baseline_6_variables,2019,0.1000,epsilon_tolerance,32,178,True,0,True,valid_partial_order



Important notes:
1. Recovery was not used as an ordering variable.
2. Epsilon > 0 uses a diagnostic tolerance rule and may create cycles.
3. Invalid partial-order runs should not be used as Hasse/POSet conclusions.
4. The safer epsilon-margin robustness check comes next.

Next notebook:
09_Epsilon_Margin_POSet_Robustness.ipynb
